In [1]:
from collections import Counter

corpus = [
    "low",
    "lower",
    "lowest",
    "low",
    "low",
    "lower"
]

# Represent each word as characters
words = [list(word) for word in corpus]

print(words)

pair_counts = Counter()

for word in words:
    for i in range(len(word) - 1):
        pair = (word[i], word[i + 1])
        pair_counts[pair] += 1

print("\nPair frequencies:")
for pair, count in pair_counts.most_common():
    print(pair, "->", count)

[['l', 'o', 'w'], ['l', 'o', 'w', 'e', 'r'], ['l', 'o', 'w', 'e', 's', 't'], ['l', 'o', 'w'], ['l', 'o', 'w'], ['l', 'o', 'w', 'e', 'r']]

Pair frequencies:
('l', 'o') -> 6
('o', 'w') -> 6
('w', 'e') -> 3
('e', 'r') -> 2
('e', 's') -> 1
('s', 't') -> 1


In [2]:
from collections import Counter

def merge_pair(words, pair_to_merge):
    merged_words = []

    for word in words:
        new_word = []
        i = 0

        while i < len(word):
            if (
                i < len(word) - 1
                and (word[i], word[i + 1]) == pair_to_merge
            ):
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1

        merged_words.append(new_word)

    return merged_words


pair_to_merge = ('l', 'o')

words = merge_pair(words, pair_to_merge)

print(words)

[['lo', 'w'], ['lo', 'w', 'e', 'r'], ['lo', 'w', 'e', 's', 't'], ['lo', 'w'], ['lo', 'w'], ['lo', 'w', 'e', 'r']]


In [3]:
pair_counts = Counter()

for word in words:
    for i in range(len(word) - 1):
        pair = (word[i], word[i + 1])
        pair_counts[pair] += 1

print("\nPair frequencies after first merge:")
for pair, count in pair_counts.most_common():
    print(pair, "->", count)


Pair frequencies after first merge:
('lo', 'w') -> 6
('w', 'e') -> 3
('e', 'r') -> 2
('e', 's') -> 1
('s', 't') -> 1


In [4]:
pair_to_merge = ("lo", "w")

words = merge_pair(words, pair_to_merge)

print(words)

[['low'], ['low', 'e', 'r'], ['low', 'e', 's', 't'], ['low'], ['low'], ['low', 'e', 'r']]


In [5]:
pair_counts = Counter()

for word in words:
    for i in range(len(word) - 1):
        pair = (word[i], word[i + 1])
        pair_counts[pair] += 1

print("\nPair frequencies after second merge:")
for pair, count in pair_counts.most_common():
    print(pair, "->", count)


Pair frequencies after second merge:
('low', 'e') -> 3
('e', 'r') -> 2
('e', 's') -> 1
('s', 't') -> 1


In [6]:
from collections import Counter

def get_pair_counts(words):
    pair_counts = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += 1

    return pair_counts


def train_bpe(words, num_merges):

    merge_rules = []

    for step in range(num_merges):

        pair_counts = get_pair_counts(words)

        if not pair_counts:
            break

        best_pair, best_count = pair_counts.most_common(1)[0]

        words = merge_pair(words, best_pair)

        merge_rules.append(best_pair)

        print(
            f"Merge {step + 1}: "
            f"{best_pair} → "
            f"{best_pair[0] + best_pair[1]} "
            f"(count={best_count})"
        )

    return words, merge_rules


# Start from original character representation
initial_words = [list(word) for word in corpus]

final_words, merge_rules = train_bpe(
    initial_words,
    num_merges=10
)

print("\nFinal words:")
print(final_words)

print("\nLearned merge rules:")
print(merge_rules)

Merge 1: ('l', 'o') → lo (count=6)
Merge 2: ('lo', 'w') → low (count=6)
Merge 3: ('low', 'e') → lowe (count=3)
Merge 4: ('lowe', 'r') → lower (count=2)
Merge 5: ('lowe', 's') → lowes (count=1)
Merge 6: ('lowes', 't') → lowest (count=1)

Final words:
[['low'], ['lower'], ['lowest'], ['low'], ['low'], ['lower']]

Learned merge rules:
[('l', 'o'), ('lo', 'w'), ('low', 'e'), ('lowe', 'r'), ('lowe', 's'), ('lowes', 't')]


In [7]:
def apply_bpe(word, merge_rules):

    tokens = list(word)

    for pair in merge_rules:

        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and (tokens[i], tokens[i + 1]) == pair
            ):
                new_tokens.append(
                    tokens[i] + tokens[i + 1]
                )
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens


test_words = [
    "low",
    "lower",
    "lowest",
    "lowers",
    "lowering"
]

for word in test_words:
    print(
        word,
        "→",
        apply_bpe(word, merge_rules)
    )

low → ['low']
lower → ['lower']
lowest → ['lowest']
lowers → ['lower', 's']
lowering → ['lower', 'i', 'n', 'g']


In [8]:
# Build a vocabulary from the tokens learned from our corpus

vocab = sorted(
    set(
        token
        for word in final_words
        for token in word
    )
)

token_to_id = {
    token: idx
    for idx, token in enumerate(vocab)
}

id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

print("Vocabulary:")
print(token_to_id)

Vocabulary:
{'low': 0, 'lower': 1, 'lowest': 2}


In [11]:
final_words

[['low'], ['lower'], ['lowest'], ['low'], ['low'], ['lower']]

In [10]:
test_words

['low', 'lower', 'lowest', 'lowers', 'lowering']

In [9]:
for word in test_words:
    bpe_tokens = apply_bpe(word, merge_rules)
    token_ids = [token_to_id[token] for token in bpe_tokens]

    print(
        word,
        "→",
        bpe_tokens,
        "→",
        token_ids
    )

low → ['low'] → [0]
lower → ['lower'] → [1]
lowest → ['lowest'] → [2]


KeyError: 's'

In [12]:
# Base vocabulary
base_vocab = list("abcdefghijklmnopqrstuvwxyz")

# Tokens created by BPE merges
merged_vocab = [
    left + right
    for left, right in merge_rules
]

# Combine and remove duplicates while preserving order
vocab = list(dict.fromkeys(
    base_vocab + merged_vocab
))

token_to_id = {
    token: idx
    for idx, token in enumerate(vocab)
}

id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

print("Vocabulary size:", len(vocab))
print(token_to_id)

Vocabulary size: 32
{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, 'lo': 26, 'low': 27, 'lowe': 28, 'lower': 29, 'lowes': 30, 'lowest': 31}


In [13]:
for word in test_words:
    bpe_tokens = apply_bpe(word, merge_rules)
    token_ids = [token_to_id[token] for token in bpe_tokens]

    print(
        word,
        "→",
        bpe_tokens,
        "→",
        token_ids
    )

low → ['low'] → [27]
lower → ['lower'] → [29]
lowest → ['lowest'] → [31]
lowers → ['lower', 's'] → [29, 18]
lowering → ['lower', 'i', 'n', 'g'] → [29, 8, 13, 6]


In [14]:
corpus = [
    "play",
    "played",
    "playing",
    "player",
    "plays",
    "replay",
    "replayed",
    "replaying",
    "talk",
    "talked",
    "talking",
    "talker",
    "walk",
    "walked",
    "walking",
    "walker"
]

words = [list(word) for word in corpus]

final_words, merge_rules = train_bpe(
    words,
    num_merges=12
)

print("\nLearned merge rules:")
for rule in merge_rules:
    print(rule)

Merge 1: ('p', 'l') → pl (count=8)
Merge 2: ('pl', 'a') → pla (count=8)
Merge 3: ('pla', 'y') → play (count=8)
Merge 4: ('a', 'l') → al (count=8)
Merge 5: ('al', 'k') → alk (count=8)
Merge 6: ('e', 'd') → ed (count=4)
Merge 7: ('i', 'n') → in (count=4)
Merge 8: ('in', 'g') → ing (count=4)
Merge 9: ('t', 'alk') → talk (count=4)
Merge 10: ('w', 'alk') → walk (count=4)
Merge 11: ('e', 'r') → er (count=3)
Merge 12: ('r', 'e') → re (count=3)

Learned merge rules:
('p', 'l')
('pl', 'a')
('pla', 'y')
('a', 'l')
('al', 'k')
('e', 'd')
('i', 'n')
('in', 'g')
('t', 'alk')
('w', 'alk')
('e', 'r')
('r', 'e')


In [16]:
# Step 1 — add a boundary marker
corpus = [
    "play well",
    "played well",
    "playing well",
    "player walks",
    "replay works",
    "talking loudly",
    "walked slowly"
]

words = []

for sentence in corpus:
    sentence_words = sentence.split()

    for word in sentence_words:
        words.append(list("▁" + word))

print(words)

[['▁', 'p', 'l', 'a', 'y'], ['▁', 'w', 'e', 'l', 'l'], ['▁', 'p', 'l', 'a', 'y', 'e', 'd'], ['▁', 'w', 'e', 'l', 'l'], ['▁', 'p', 'l', 'a', 'y', 'i', 'n', 'g'], ['▁', 'w', 'e', 'l', 'l'], ['▁', 'p', 'l', 'a', 'y', 'e', 'r'], ['▁', 'w', 'a', 'l', 'k', 's'], ['▁', 'r', 'e', 'p', 'l', 'a', 'y'], ['▁', 'w', 'o', 'r', 'k', 's'], ['▁', 't', 'a', 'l', 'k', 'i', 'n', 'g'], ['▁', 'l', 'o', 'u', 'd', 'l', 'y'], ['▁', 'w', 'a', 'l', 'k', 'e', 'd'], ['▁', 's', 'l', 'o', 'w', 'l', 'y']]


In [20]:
text = "café 🚀"

print(text.encode("utf-8"))

b'caf\xc3\xa9 \xf0\x9f\x9a\x80'


In [18]:
texts = [
    "hello",
    "café",
    "नमस्ते",
    "你好",
    "🚀"
]

for text in texts:
    encoded = text.encode("utf-8")

    print(f"\n{text}")
    print("Bytes:", list(encoded))
    print("Number of bytes:", len(encoded))


hello
Bytes: [104, 101, 108, 108, 111]
Number of bytes: 5

café
Bytes: [99, 97, 102, 195, 169]
Number of bytes: 5

नमस्ते
Bytes: [224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135]
Number of bytes: 18

你好
Bytes: [228, 189, 160, 229, 165, 189]
Number of bytes: 6

🚀
Bytes: [240, 159, 154, 128]
Number of bytes: 4


In [21]:
texts = [
    "hello hello",
    "hello world",
    "hello there",
    "café café"
]

byte_words = []

for text in texts:
    tokens = [
        bytes([b])
        for b in text.encode("utf-8")
    ]
    byte_words.append(tokens)

for text, tokens in zip(texts, byte_words):
    print(f"\n{text}")
    print([list(token) for token in tokens])


hello hello
[[104], [101], [108], [108], [111], [32], [104], [101], [108], [108], [111]]

hello world
[[104], [101], [108], [108], [111], [32], [119], [111], [114], [108], [100]]

hello there
[[104], [101], [108], [108], [111], [32], [116], [104], [101], [114], [101]]

café café
[[99], [97], [102], [195], [169], [32], [99], [97], [102], [195], [169]]


In [22]:
pair_counts = get_pair_counts(byte_words)

print("\nMost frequent byte pairs:")

for pair, count in pair_counts.most_common(10):
    print(
        [list(pair[0]), list(pair[1])],
        "->",
        count
    )


Most frequent byte pairs:
[[104], [101]] -> 5
[[101], [108]] -> 4
[[108], [108]] -> 4
[[108], [111]] -> 4
[[111], [32]] -> 3
[[99], [97]] -> 2
[[97], [102]] -> 2
[[102], [195]] -> 2
[[195], [169]] -> 2
[[32], [104]] -> 1


In [23]:
best_pair, best_count = pair_counts.most_common(1)[0]

print(
    "\nMerging:",
    [list(best_pair[0]), list(best_pair[1])],
    "count:",
    best_count
)

byte_words = merge_pair(
    byte_words,
    best_pair
)

print("\nAfter merge:")
for word in byte_words:
    print([list(token) for token in word])


Merging: [[104], [101]] count: 5

After merge:
[[104, 101], [108], [108], [111], [32], [104, 101], [108], [108], [111]]
[[104, 101], [108], [108], [111], [32], [119], [111], [114], [108], [100]]
[[104, 101], [108], [108], [111], [32], [116], [104, 101], [114], [101]]
[[99], [97], [102], [195], [169], [32], [99], [97], [102], [195], [169]]


In [44]:
byte_final_words, byte_merge_rules = train_bpe(
    byte_words,
    num_merges=10
)

print("\nLearned byte-level merge rules:")

for left, right in byte_merge_rules:
    print(
        list(left),
        "+",
        list(right),
        "→",
        list(left + right)
    )

Merge 1: (b'he', b'l') → b'hel' (count=4)
Merge 2: (b'hel', b'l') → b'hell' (count=4)
Merge 3: (b'hell', b'o') → b'hello' (count=4)
Merge 4: (b'hello', b' ') → b'hello ' (count=3)
Merge 5: (b'c', b'a') → b'ca' (count=2)
Merge 6: (b'ca', b'f') → b'caf' (count=2)
Merge 7: (b'caf', b'\xc3') → b'caf\xc3' (count=2)
Merge 8: (b'caf\xc3', b'\xa9') → b'caf\xc3\xa9' (count=2)
Merge 9: (b'hello ', b'hello') → b'hello hello' (count=1)
Merge 10: (b'hello ', b'w') → b'hello w' (count=1)

Learned byte-level merge rules:
[104, 101] + [108] → [104, 101, 108]
[104, 101, 108] + [108] → [104, 101, 108, 108]
[104, 101, 108, 108] + [111] → [104, 101, 108, 108, 111]
[104, 101, 108, 108, 111] + [32] → [104, 101, 108, 108, 111, 32]
[99] + [97] → [99, 97]
[99, 97] + [102] → [99, 97, 102]
[99, 97, 102] + [195] → [99, 97, 102, 195]
[99, 97, 102, 195] + [169] → [99, 97, 102, 195, 169]
[104, 101, 108, 108, 111, 32] + [104, 101, 108, 108, 111] → [104, 101, 108, 108, 111, 32, 104, 101, 108, 108, 111]
[104, 101, 108,

### Real tokenizer used by SmolLM2-135M

In [46]:
from transformers import AutoTokenizer

model_name = "huggingFaceTB/smolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)
print("UNK token:", tokenizer.unk_token)
print("UNK ID:", tokenizer.unk_token_id)

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Tokenizer class: GPT2Tokenizer
Vocabulary size: 49152
EOS token: <|im_end|>
EOS ID: 2
UNK token: <|endoftext|>
UNK ID: 0


In [47]:
text = 'prateek arrived'
ids = tokenizer.encode(text, add_special_tokens=False)
tokens = tokenizer.convert_ids_to_tokens(ids)

print('Text:', text)
print('Tokens:', tokens)
print('IDs:', ids)
print('Token count:', len(ids))

Text: prateek arrived
Tokens: ['pr', 'ate', 'ek', 'Ġarrived']
IDs: [1180, 368, 1846, 6612]
Token count: 4


In [48]:
texts = [
    "prateek arrived",
    "unhappiness",
    "replaying",
    "Hello, how are you?",
    "machine-learning",
    "café",
    "नमस्ते",
    "你好",
    "🚀",
    "This is a very longwordthatwasneverseenbefore."
]

for text in texts:
    ids = tokenizer.encode(text, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(ids)

    print("\nText:", text)
    print("Tokens:", tokens)
    print("IDs:", ids)
    print("Token Count:", len(ids))


Text: prateek arrived
Tokens: ['pr', 'ate', 'ek', 'Ġarrived']
IDs: [1180, 368, 1846, 6612]
Token Count: 4

Text: unhappiness
Tokens: ['un', 'h', 'appiness']
IDs: [419, 88, 38462]
Token Count: 3

Text: replaying
Tokens: ['re', 'playing']
IDs: [257, 26517]
Token Count: 2

Text: Hello, how are you?
Tokens: ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']
IDs: [19556, 28, 638, 359, 346, 47]
Token Count: 6

Text: machine-learning
Tokens: ['machine', '-', 'learning']
IDs: [24939, 29, 13193]
Token Count: 3

Text: café
Tokens: ['c', 'af', 'Ã©']
IDs: [83, 1939, 2756]
Token Count: 3

Text: नमस्ते
Tokens: ['à¤¨', 'à¤®', 'à¤¸', 'à¥į', 'à¤¤', 'à¥ĩ']
IDs: [30658, 35090, 38770, 16525, 26293, 30581]
Token Count: 6

Text: 你好
Tokens: ['ä½', 'ł', 'å¥', '½']
IDs: [18645, 250, 48392, 138]
Token Count: 4

Text: 🚀
Tokens: ['ðŁ', 'ļ', 'Ģ']
IDs: [10813, 244, 218]
Token Count: 3

Text: This is a very longwordthatwasneverseenbefore.
Tokens: ['This', 'Ġis', 'Ġa', 'Ġvery', 'Ġlong', 'word', 'that', 'was', 'ne', 'verse'

In [51]:
import sys
sys.path.append("..")

from tokenizer import Tokenizer

tokenizer_old = Tokenizer()

with open("../data/sample.txt", "r") as file:
    text = file.read()

tokenizer_old.build_vocab(text)

In [53]:
text = "prateek arrived in Willow Creek."

our_ids = tokenizer_old.encode(text)
real_ids = tokenizer.encode(
    text,
    add_special_tokens=False
)

print("Our tokenizer:")
print(tokenizer_old.decode(our_ids))
print("\nOur IDs:")
print(our_ids)
print("\nOur token count:")
print(len(our_ids))

print("Real tokenizer:")
print(tokenizer.convert_ids_to_tokens(real_ids))

print("\nReal IDs:")
print(real_ids)

print("\nReal token count:")
print(len(real_ids))

Our tokenizer:
prateek arrived in willow creek.

Our IDs:
[366, 32, 227, 560, 110, 6]

Our token count:
6
Real tokenizer:
['pr', 'ate', 'ek', 'Ġarrived', 'Ġin', 'ĠWillow', 'ĠCreek', '.']

Real IDs:
[1180, 368, 1846, 6612, 281, 45974, 10896, 30]

Real token count:
8
